In [2]:
from dotenv import load_dotenv
import os
load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY")


In [3]:
from langchain_community import document_loaders
from langchain_community.document_loaders import TextLoader
loader=TextLoader('my_document.txt')
document=loader.load()

/tmp/ipykernel_22659/2363140360.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community import document_loaders
/home/vasim/Projects/projects/rag_with_langchain/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
documents_chunk=splitter.split_documents(document)

In [5]:
from huggingface_hub import snapshot_download
MODEL_PATH = "./models/all-MiniLM-L6-v2"

# Only download if not already present
if not os.path.exists(MODEL_PATH):
    snapshot_download(
        repo_id="sentence-transformers/all-MiniLM-L6-v2",
        local_dir=MODEL_PATH
    )

Fetching 30 files: 100%|██████████| 30/30 [04:27<00:00,  8.90s/it]


In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')


/tmp/ipykernel_22659/1190950895.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4989.76it/s]


In [7]:
from langchain_community.vectorstores import FAISS
vector_store=FAISS.from_documents(documents_chunk,embeddings)

In [8]:
retriever=vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)

In [9]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [10]:
llm=ChatGroq(model="llama-3.1-8b-instant",api_key=groq_api_key)


In [11]:

#prompt
prompt=ChatPromptTemplate.from_template(
    """ 
    Answer the question based on the context below only.
    If you don't know, say you don't know.

    context:{context}
    Question:{question}
    """
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

#Chain

chain=(
    {
        "context":retriever | format_docs,
    "question":RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()

)


response=chain.invoke("How many spiderman movies are there till date?")
print(response)

I can answer that. 

According to the context, there are three live-action Spider-Man movie trilogies: 

1. Sam Raimi's Spider-Man trilogy (2002-2007) starring Tobey Maguire.
2. The Amazing Spider-Man duology (2012-2014) starring Andrew Garfield.
3. The Marvel Cinematic Universe (MCU) Spider-Man movies (2016-present) starring Tom Holland.

Additionally, there are multiple animated and other forms of Spider-Man movies, but the question specifically asks about live-action movies. Therefore, the total number of live-action Spider-Man movies is 6 (3 trilogies and 2 duologies, but the 2 duologies is actually 2 movies, plus another 4 movies of the MCU trilogy, making it 6).
